# 04 - PydanticAI: Compliance Caseworker

## Scenario: Northstar Compliance Auditing

In a compliance environment (like GDPR data requests or SOC2 audits), you cannot rely on unstructured text. You need strict data typing. If an LLM hallucinates a boolean or outputs a malformed JSON string, your auditing pipeline will crash.

**PydanticAI** is an agent framework built *on top* of Pydantic. It treats the LLM as a function that strictly returns Pydantic models, and natively injects Pydantic schemas into tool calls.

In this notebook, we will build a GDPR Data Request caseworker that extracts specific fields and validates them strictly.

In [1]:
# You may need to install pydantic-ai in your environment
# !pip install pydantic-ai


In [2]:
import os
from pydantic import BaseModel, Field
# We wrap the import in a try/except so the notebook can still run if the package is missing
try:
    from pydantic_ai import Agent, RunContext
    HAS_PYDANTIC_AI = True
except ImportError:
    HAS_PYDANTIC_AI = False
    print("pydantic-ai is not installed. We will mock the output.")

# 1. Define the strict output schema
class GDPRRequest(BaseModel):
    user_email: str = Field(description="The email address of the requester")
    request_type: str = Field(description="Must be 'delete' or 'export'")
    urgency_days: int = Field(description="Number of days remaining to comply (default 30)")

# 2. Define the Agent
if HAS_PYDANTIC_AI:
    try:
        # Notice we define `result_type=GDPRRequest`. PydanticAI forces the LLM to return this model.
        agent = Agent(
            'openai:gpt-4o',
            result_type=GDPRRequest,
            system_prompt="You are a compliance caseworker. Extract the GDPR request details from the ticket."
        )
    except TypeError:
        # Fallback if an older version of pydantic-ai is installed
        HAS_PYDANTIC_AI = False
        print("Older version of pydantic-ai detected. Mocking agent.")


Older version of pydantic-ai detected. Mocking agent.


## 1. Running the Agent

When we run the agent, the output is not a string. It is guaranteed to be a `GDPRRequest` Python object.

In [3]:
ticket_text = """
    Hello,
    I live in the EU and want all my data removed from your servers. 
    My email is angry.customer@example.com. You have 7 days before I contact regulators.
    """

if HAS_PYDANTIC_AI:
    # Run the agent
    # We pass a dummy API key to avoid crashing if you don't have one configured
    os.environ.setdefault("OPENAI_API_KEY", "dummy-key")
    try:
        result = agent.run_sync(ticket_text)
        print("Data Extracted Successfully!")
        print(f"Email: {result.data.user_email}")
        print(f"Type:  {result.data.request_type}")
        print(f"Days:  {result.data.urgency_days}")
    except Exception as e:
        print(f"Agent Execution Failed (Check API Key): {e}")
else:
    print("Data Extracted Successfully (MOCK)!")
    print("Email: angry.customer@example.com")
    print("Type:  delete")
    print("Days:  7")


Data Extracted Successfully (MOCK)!
Email: angry.customer@example.com
Type:  delete
Days:  7


## 2. Dynamic Tool Dependencies

In PydanticAI, tools can access context (like a database connection) without passing it in the LLM prompt.

In [4]:
if HAS_PYDANTIC_AI:
    from dataclasses import dataclass
    
    @dataclass
    class DBConnection:
        dsn: str
        
    # Agent with dependency injection
    try:
        db_agent = Agent(
            'openai:gpt-4o',
            deps_type=DBConnection,
            result_type=bool,
            system_prompt="Check if the user exists."
        )
        
        @db_agent.tool
        def check_user_exists(ctx: RunContext[DBConnection], email: str) -> bool:
            # The LLM doesn't see 'ctx'. The application provides it at runtime!
            print(f"Connecting to {ctx.deps.dsn} to check {email}...")
            return True

        result = db_agent.run_sync("Does angry.customer@example.com exist?", deps=DBConnection(dsn="postgres://..."))
        print(f"User exists: {result.data}")
    except TypeError:
        print("Older version of pydantic-ai detected. Mocking agent execution.")
    except Exception as e:
        print(f"Execution Failed: {e}")


## Watch For

- **Strictness vs Flexibility**: PydanticAI is excellent when you need guaranteed structured data. But if you want a conversational chatbot, a framework like LangChain or AutoGen might be more suitable.
- **Validation Errors**: If the LLM hallucinates an integer instead of a string, PydanticAI automatically catches the `ValidationError` and asks the LLM to fix it.

## Checkpoint

**1. What happens in PydanticAI if the LLM returns `"urgency_days": "seven"` (a string instead of an int)?**
- A) The program crashes immediately with a KeyError.
- B) Pydantic automatically catches the validation error, sends it back to the LLM, and asks it to correct the schema.
- C) It converts it to `0`.
- D) It ignores the schema completely.
